# Get Nadex Contract Locations

Downloads daily settlement PDFs from Nadex's public S3 bucket, extracts US 500 binary
contract strikes, and identifies the strikes that bracket each day's 09:30 market open.

**Output:** `contract_locations.csv` — columns: `date`, `above`, `below`

**Run before `barsToCleaning.ipynb`** whenever new trading dates have been added to `GoodOldGoodOld.csv`.

> PDFs are settlement results, available after market close.  
> Same-day (live) contract lookup is not supported by this script.

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
# Filters to isolate US 500 daily binary options expiring at 4:15 PM ET
# Note: only PRODUCT_FILTER and EXPIRY_FILTER are used in text scanning.
# The 4:15PM expiry is unique to Daily contracts — no PERIOD_FILTER needed.
PRODUCT_FILTER = "US 500"
EXPIRY_FILTER  = "4:15PM"

PDF_URL     = "https://s3.amazonaws.com/market-data-prod.nadex.com/{date}_tradingResults.pdf"
OUTPUT_FILE = "contract_locations.csv"
BARS_FILE   = "GoodOldGoodOld.csv"
# ──────────────────────────────────────────────────────────────────────────────

In [ ]:
import re
import io
import logging
import requests
import pandas as pd
import pdfplumber
from concurrent.futures import ThreadPoolExecutor, as_completed

# Suppress harmless pdfminer decompression warnings (noise from PDF internal structure)
logging.getLogger('pdfminer').setLevel(logging.ERROR)

In [ ]:
# Load 1-minute bars and extract the 09:30:00 open price for each trading day
bars = pd.read_csv(BARS_FILE)
bars['date_only'] = pd.to_datetime(bars['date']).dt.normalize()

opens = (
    bars[bars['time'] == '09:30:00']
    .groupby('date_only')['open']
    .first()
    .reset_index()
    .rename(columns={'date_only': 'date', 'open': 'market_open'})
)

print(f"Trading days in {BARS_FILE}: {len(opens)}")
print(opens.tail())

In [ ]:
# Load existing contract_locations.csv for incremental updates.
# Dates already covered are skipped — no redundant PDF downloads.
try:
    existing = pd.read_csv(OUTPUT_FILE, parse_dates=['date'])
    existing['date'] = existing['date'].dt.normalize()
    covered_dates = set(existing['date'])
    print(f"Existing {OUTPUT_FILE} covers {len(covered_dates)} date(s)")
except FileNotFoundError:
    existing = pd.DataFrame(columns=['date', 'above', 'below'])
    covered_dates = set()
    print(f"No existing {OUTPUT_FILE} — building from scratch")

dates_to_fetch = [d for d in opens['date'] if d not in covered_dates]
print(f"Dates to fetch: {len(dates_to_fetch)}")

In [ ]:
# Regex pattern for Display Name formats:
#   "US 500 (Jun) >5004.0 (4:15PM)"  or  "US 500 (Jun) +4823.6 (4:15PM)"
# Both '>' and '+' appear in Nadex PDFs depending on PDF generation version.
_STRIKE_RE = re.compile(r'US 500[^>+\d]*[>+]([\d]{4,5}(?:\.\d+)?)')


def _parse_strikes_from_pdf(pdf_bytes):
    """Return all US 500 daily binary strike prices found in a Nadex results PDF."""
    strikes = []
    with pdfplumber.open(pdf_bytes) as pdf:
        for page in pdf.pages:
            text = page.extract_text() or ''
            for line in text.splitlines():
                if PRODUCT_FILTER not in line or EXPIRY_FILTER not in line:
                    continue
                m = _STRIKE_RE.search(line)
                if m:
                    strikes.append(float(m.group(1)))
    return strikes


def get_contracts_for_date(date, market_open):
    """Download the Nadex results PDF for `date` and return (above_strike, below_strike).

    Returns (None, None) if the PDF is unavailable (weekend / holiday / future date)
    or a network error occurs.
    """
    url = PDF_URL.format(date=date.strftime('%Y%m%d'))
    try:
        resp = requests.get(url, timeout=30)
    except requests.RequestException as e:
        print(f'  {date.date()} network error: {e}')
        return None, None

    if resp.status_code != 200:
        return None, None  # Weekend, holiday, or PDF not yet published

    strikes = _parse_strikes_from_pdf(io.BytesIO(resp.content))
    if not strikes:
        print(f'  {date.date()} WARNING: PDF downloaded but no US 500 daily strikes found')
        return None, None

    strikes = sorted(set(strikes))
    belows  = [s for s in strikes if s <= market_open]
    aboves  = [s for s in strikes if s >  market_open]
    below   = max(belows) if belows else None
    above   = min(aboves) if aboves else None

    if above is None:
        print(f'  {date.date()} WARNING: market open {market_open} is above all strikes')
    if below is None:
        print(f'  {date.date()} WARNING: market open {market_open} is below all strikes')

    return above, below

In [ ]:
# Fetch contract data for all missing dates in parallel and save to contract_locations.csv.
# 12 workers — S3 HTTP requests are I/O-bound; threading is safe and ~10x faster than serial.
market_opens = dict(zip(opens['date'], opens['market_open']))

def _fetch(date):
    above, below = get_contracts_for_date(date, market_opens[date])
    return date, above, below

rows = []
print(f"Fetching {len(dates_to_fetch)} date(s) with 12 parallel workers...")

with ThreadPoolExecutor(max_workers=12) as executor:
    futures = {executor.submit(_fetch, d): d for d in dates_to_fetch}
    for future in as_completed(futures):
        date, above, below = future.result()
        if above is not None or below is not None:
            rows.append({'date': date, 'above': above, 'below': below})
            print(f"{date.date()}  open={market_opens[date]:.2f}  below={below}  above={above}")
        else:
            print(f"{date.date()}  — no data (holiday / weekend / future date)")

new_data = pd.DataFrame(rows)
combined = (
    pd.concat([existing, new_data], ignore_index=True)
    .drop_duplicates('date')
    .sort_values('date')
    .reset_index(drop=True)
)
combined.to_csv(OUTPUT_FILE, index=False)
print(f"\nSaved {len(combined)} date(s) to {OUTPUT_FILE}")